In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os

base_path = "/content/drive/MyDrive/files"

folders = [f for f in os.listdir(base_path) if f.startswith("S")]

filenumber = ['03','07','11']
data = []

for folder in sorted(folders):
    folder_path = os.path.join(base_path, folder)

    if not os.path.isdir(folder_path):
        continue

    files = os.listdir(folder_path)

    for f in files:
        if f.lower().endswith(".edf"):

            run_part = f.split("R")[-1].split(".")[0]

            if run_part in filenumber:
                full_path = os.path.join(folder_path, f)
                data.append(full_path)

# Print final single list
print(data)
print("\nTotal files:", len(data))


['/content/drive/MyDrive/files/S071/S071R03.edf', '/content/drive/MyDrive/files/S071/S071R11.edf', '/content/drive/MyDrive/files/S071/S071R07.edf', '/content/drive/MyDrive/files/S072/S072R11.edf', '/content/drive/MyDrive/files/S072/S072R07.edf', '/content/drive/MyDrive/files/S072/S072R03.edf', '/content/drive/MyDrive/files/S074/S074R07.edf', '/content/drive/MyDrive/files/S074/S074R03.edf', '/content/drive/MyDrive/files/S074/S074R11.edf', '/content/drive/MyDrive/files/S076/S076R03.edf', '/content/drive/MyDrive/files/S076/S076R11.edf', '/content/drive/MyDrive/files/S076/S076R07.edf', '/content/drive/MyDrive/files/S077/S077R03.edf', '/content/drive/MyDrive/files/S077/S077R07.edf', '/content/drive/MyDrive/files/S077/S077R11.edf', '/content/drive/MyDrive/files/S078/S078R11.edf', '/content/drive/MyDrive/files/S078/S078R07.edf', '/content/drive/MyDrive/files/S078/S078R03.edf', '/content/drive/MyDrive/files/S079/S079R07.edf', '/content/drive/MyDrive/files/S079/S079R03.edf', '/content/drive/MyD

In [9]:
!pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 93.0 MB/s eta 0:00:00


In [10]:
import mne

raw_list = []

for path in data:
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    raw_list.append(raw)

print("Total raw files loaded:", len(raw_list))

/tmp/ipython-input-235/796524558.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipython-input-235/796524558.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipython-input-235/796524558.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)


Total raw files loaded: 111


In [19]:
import numpy as np
import mne

results = {}

for i, raw in enumerate(raw_list):

    subject_key = f"subject_{i+1}"
    results[subject_key] = {}

    events, event_dict = mne.events_from_annotations(raw, verbose=False)

    epochs = mne.Epochs(
        raw,
        events,
        event_id={'T0': 1, 'T1': 2, 'T2': 3},
        tmin=0.5,
        tmax=2.5,
        baseline=None,
        preload=True,
        verbose=False
    )

    picks = mne.pick_channels(epochs.ch_names, include=['C3..', 'C4..'])

    # =====================================================
    # 🔵 MU BAND
    # =====================================================
    epochs_mu = epochs.copy().filter(8., 13., verbose=False)

    # ---- T0 baseline (mu band)
    data_t0_mu = epochs_mu['T0'].get_data()[:, picks, :]
    power_t0_mu = np.mean(data_t0_mu ** 2, axis=2)
    mean_power_t0_mu = np.mean(power_t0_mu, axis=0)

    results[subject_key]['T0_mu_mean_power'] = {
        'C3': mean_power_t0_mu[0],
        'C4': mean_power_t0_mu[1]
    }
    for task in ['T1', 'T2']:
        data_mu = epochs_mu[task].get_data()[:, picks, :]
        var_mu = np.var(data_mu, axis=2)

        results[subject_key][f'{task}_mu_variances'] = {
            'C3': var_mu[:, 0].tolist(),
            'C4': var_mu[:, 1].tolist()
        }
    epochs_beta = epochs.copy().filter(13., 30., verbose=False)
    data_t0_beta = epochs_beta['T0'].get_data()[:, picks, :]
    power_t0_beta = np.mean(data_t0_beta ** 2, axis=2)
    mean_power_t0_beta = np.mean(power_t0_beta, axis=0)

    results[subject_key]['T0_beta_mean_power'] = {
        'C3': mean_power_t0_beta[0],
        'C4': mean_power_t0_beta[1]
    }
    for task in ['T1', 'T2']:
        data_beta = epochs_beta[task].get_data()[:, picks, :]
        var_beta = np.var(data_beta, axis=2)

        results[subject_key][f'{task}_beta_variances'] = {
            'C3': var_beta[:, 0].tolist(),
            'C4': var_beta[:, 1].tolist()
        }

In [20]:
import numpy as np

save_path = "/content/drive/MyDrive/bci_results.npy"

np.save(save_path, results, allow_pickle=True)

print("Results saved as NumPy file!")

Results saved as NumPy file!


In [27]:
import numpy as np

sub = results["subject_10"]

# ==========================
# 🔵 MU BAND ERD
# ==========================

C3_rest_mu = sub["T0_mu_mean_power"]["C3"]
C4_rest_mu = sub["T0_mu_mean_power"]["C4"]

C3_t1_mu = np.array(sub["T1_mu_variances"]["C3"])
C4_t1_mu = np.array(sub["T1_mu_variances"]["C4"])

C3_t2_mu = np.array(sub["T2_mu_variances"]["C3"])
C4_t2_mu = np.array(sub["T2_mu_variances"]["C4"])

ERD_C3_t1_mu = (C3_t1_mu - C3_rest_mu) / C3_rest_mu
ERD_C4_t1_mu = (C4_t1_mu - C4_rest_mu) / C4_rest_mu

ERD_C3_t2_mu = (C3_t2_mu - C3_rest_mu) / C3_rest_mu
ERD_C4_t2_mu = (C4_t2_mu - C4_rest_mu) / C4_rest_mu

print("==== MU ERD ====")
print("Mean ERD T1 - C3:", np.mean(ERD_C3_t1_mu))
print("Mean ERD T1 - C4:", np.mean(ERD_C4_t1_mu))
print("Mean ERD T2 - C3:", np.mean(ERD_C3_t2_mu))
print("Mean ERD T2 - C4:", np.mean(ERD_C4_t2_mu))


# ==========================
# 🔵 BETA BAND ERD
# ==========================

C3_rest_beta = sub["T0_beta_mean_power"]["C3"]
C4_rest_beta = sub["T0_beta_mean_power"]["C4"]

C3_t1_beta = np.array(sub["T1_beta_variances"]["C3"])
C4_t1_beta = np.array(sub["T1_beta_variances"]["C4"])

C3_t2_beta = np.array(sub["T2_beta_variances"]["C3"])
C4_t2_beta = np.array(sub["T2_beta_variances"]["C4"])

ERD_C3_t1_beta = (C3_t1_beta - C3_rest_beta) / C3_rest_beta
ERD_C4_t1_beta = (C4_t1_beta - C4_rest_beta) / C4_rest_beta

ERD_C3_t2_beta = (C3_t2_beta - C3_rest_beta) / C3_rest_beta
ERD_C4_t2_beta = (C4_t2_beta - C4_rest_beta) / C4_rest_beta

print("\n==== BETA ERD ====")
print("Mean ERD T1 - C3:", np.mean(ERD_C3_t1_beta))
print("Mean ERD T1 - C4:", np.mean(ERD_C4_t1_beta))
print("Mean ERD T2 - C3:", np.mean(ERD_C3_t2_beta))
print("Mean ERD T2 - C4:", np.mean(ERD_C4_t2_beta))


==== MU ERD ====
Mean ERD T1 - C3: -0.11591218084468077
Mean ERD T1 - C4: -0.04934338560754619
Mean ERD T2 - C3: -0.28467147724208125
Mean ERD T2 - C4: -0.2858254252505598

==== BETA ERD ====
Mean ERD T1 - C3: -0.17528160234601825
Mean ERD T1 - C4: -0.05590483358959855
Mean ERD T2 - C3: -0.30142816859818555
Mean ERD T2 - C4: -0.151689761999736
